# 온도 예측 잔차의 요인 분해 — 선형 기여도(①) + SHAP(②)

`ARX_Temp_Model.py` 의 예측(선형)과 실제가 크게 벌어지는 이벤트들에 대해,
**heat/exchanger/feed 중 어느 요인이 그 차이를 만들었는지** 두 가지 방법으로
분해한다. 오토인코더 대신 이 둘을 고른 이유: 입력이 사실상 3~4개 저차원
이진값이라 — 오토인코더가 배울 만큼 복잡한 "정상 매니폴드"가 없다. 저차원
표형 데이터에서 요인 기여도를 분해하는 표준 도구는 선형계수 분해와 SHAP 다.

## 두 방법이 서로 다른 것을 알려준다

| | 방법 | 알려주는 것 |
|---|---|---|
| **①** | 선형(ARX) 계수 × 입력값 | "선형 모델이 이 예측을 어떤 비중으로 만들었나" |
| **②** | 그래디언트 부스팅 + SHAP | "비선형/상호작용까지 고려하면 실제로 뭐가 더 크게 작용했나" |

**핵심 아이디어**: 같은 이벤트에서 `SHAP 기여도 − 선형 기여도` 의 차이가 큰
요인이 있다면, **그게 선형 모델이 놓친 부분**이다. 부스팅의 잔차가 선형보다
훨씬 작다면 그 차이(gap)를 요인별로 나눠준 게 바로 이 값이다.

**주의**: 두 방법 다 "모델이 무엇에 의존해 예측했나"를 알려주는 것이지, 실제
물리적 인과를 증명하지 않는다. 두 모델 다 잔차가 크게 남는 이벤트가 있다면,
그건 지금 갖고 있는 4개 요인(T, heat, exchanger, feed) 밖의 원인일 수 있다.

## 0. 설정 — ARX 모델을 그대로 재사용한다

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
import shap

# 이 노트북은 parkkt/code_ongoing/ 에서 실행된다고 가정한다.
# ARX_Temp_Model.py 는 이 폴더에, Tank_Temp_Factor_Analysis.py 는 한 단계 위(parkkt/)에 있다.
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.getcwd()))
import ARX_Temp_Model as ARX          # build_frame, ols, fit_eval, MODEL_B, STEPS, TEST_FRACTION 재사용
from Tank_Temp_Factor_Analysis import TANKS

# ARX_Temp_Model.SAVE_DIR/OUT_DIR 는 자기 파일 기준(parkkt/code_ongoing/)으로 잡혀 있어
# 실제 데이터 폴더(parkkt/extracted_csv, parkkt/analysis_out)를 가리키도록 보정한다.
ARX.SAVE_DIR = os.path.join(os.path.dirname(os.getcwd()), "extracted_csv")
ARX.OUT_DIR = os.path.join(os.path.dirname(os.getcwd()), "analysis_out")

import matplotlib
import matplotlib.pyplot as plt
# ↑ ARX_Temp_Model import 가 백엔드를 Agg 로 바꿔놓으므로 inline 은 반드시 그 뒤에
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

pd.set_option("display.width", 220)

FEATURES = ARX.MODEL_B   # ["T", "heat_fut", "heat_past", "exch_fut", "exch_past", "feed_fut", "feed_past"]
print("사용 특징:", FEATURES)

## 1. 부스팅 모델 학습 함수

`ARX.fit_eval` 과 **동일한 시간순 분할**(앞 70%/뒤 30%, 경계 purge)을 그대로
써서 두 모델이 완전히 같은 학습/테스트 행을 보게 만든다 — 그래야 이벤트
인덱스가 그대로 대응된다.

`shap.TreeExplainer` 는 트리 구조를 직접 읽어 계산하는 **정확한(근사가 아닌)**
방법이라 이 정도 데이터 크기에서는 충분히 빠르다.

In [ ]:
def fit_gb(df, features, max_iter=300, max_depth=4, random_state=0):
    n = len(df)
    split = int(n * (1 - ARX.TEST_FRACTION))
    train = df.iloc[: split - ARX.STEPS]       # ARX.fit_eval 과 동일한 purge
    test = df.iloc[split:]

    Xtr, Xte = train[features].values, test[features].values
    ytr, yte = train["y"].values, test["y"].values

    model = HistGradientBoostingRegressor(max_iter=max_iter, max_depth=max_depth,
                                          random_state=random_state)
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)

    rmse = float(np.sqrt(np.mean((yte - pred) ** 2)))
    r2 = float(1 - np.sum((yte - pred) ** 2) / np.sum((yte - yte.mean()) ** 2))

    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(Xte)

    return {"model": model, "features": features, "rmse": rmse, "r2": r2,
            "pred": pred, "actual": yte, "test_index": test.index, "test_df": test,
            "shap": shap_vals, "base_value": float(np.asarray(explainer.expected_value).ravel()[0])}

## 2. 8개 탱크 전부 — 선형 vs 부스팅 성능 비교

**부스팅이 선형보다 실제로 나아야 이 분석이 의미가 있다.** 안 나으면 비선형/
상호작용 효과가 없다는 뜻이라, 잔차는 그냥 이 4개 요인 밖의 무언가다.

In [ ]:
FITS = {}
rows = []
for tk in TANKS:
    df, _ = ARX.build_frame(tk)
    if len(df) < 2000:
        print(f"⚠️ [{tk}] 표본 부족 — 건너뜀")
        continue
    lin = ARX.fit_eval(df, FEATURES)
    gb = fit_gb(df, FEATURES)
    FITS[tk] = (lin, gb)
    rows.append({"탱크": tk, "테스트n": lin["n_test"],
                 "선형_R2": round(lin["r2"], 3), "부스팅_R2": round(gb["r2"], 3),
                 "선형_RMSE": round(lin["rmse"], 3), "부스팅_RMSE": round(gb["rmse"], 3),
                 "RMSE개선%": round((1 - gb["rmse"] / lin["rmse"]) * 100, 1)})

cmp_table = pd.DataFrame(rows)
cmp_table

## 3. 방법 ① — 선형 기여도 분해

```
선형예측 = const + Σ (계수 × 입력값)
```
각 항이 그 요인이 "선형 모델의 예측에 기여한 양"이다. 이건 이벤트마다 결정론적으로
정해지고, 계수는 전체 학습 구간에서 한 번 고정된 값이다.

In [ ]:
def linear_contrib(lin_fit, i):
    """테스트 세트 i번째 행에서 선형모델의 요인별 기여도(딕셔너리)."""
    beta, feats = lin_fit["beta"], lin_fit["features"]      # feats[0] == "const"
    row = lin_fit["test_df"].iloc[i]
    out = {"const": float(beta[0])}
    for j, f in enumerate(feats[1:]):
        out[f] = float(beta[j + 1] * row[f])
    return out


TANK = "I3"
lin, gb = FITS[TANK]

demo_i = int(np.argmax(np.abs(lin["actual"] - lin["pred"])))
print(f"[{TANK}] 선형 잔차가 가장 큰 이벤트: {lin['test_index'][demo_i]}")
print("  선형 기여도 분해:", {k: round(v, 2) for k, v in linear_contrib(lin, demo_i).items()})
print(f"  합계 = {sum(linear_contrib(lin, demo_i).values()):.2f}  (선형예측값 {lin['pred'][demo_i]:.2f} 과 같아야 함)")

## 4. 방법 ② — SHAP 기여도, 그리고 "선형이 놓친 부분"

```
부스팅예측 = base_value + Σ SHAP_i
```
SHAP 도 마찬가지로 요인별 기여도로 분해되는데, 부스팅 모델은 비선형·상호작용을
잡을 수 있어서 같은 요인이라도 상황에 따라 기여도가 달라질 수 있다.

```
차이_i = SHAP_i − 선형기여도_i
```
이 값이 큰 요인이 **"선형 모델이 그 요인의 효과를 잘못 계산했을 가능성이
큰 곳"** 이다. 부스팅 잔차가 선형 잔차보다 작을 때만 이 차이가 실제로
"설명력을 되찾은 양"이라는 의미를 갖는다.

In [ ]:
def attribution_report(tank, top_k=15):
    lin, gb = FITS[tank]
    feats = gb["features"]
    lin_resid = lin["actual"] - lin["pred"]
    gb_resid = gb["actual"] - gb["pred"]

    order = np.argsort(-np.abs(lin_resid))[:top_k]
    rows, diff_rows = [], []
    for i in order:
        lc = linear_contrib(lin, i)
        shap_row = gb["shap"][i]
        diffs = {f: float(shap_row[j] - lc[f]) for j, f in enumerate(feats)}
        # T 는 액추에이터가 아니라 상태값(현재 온도)이다. 선형모델은 T 의 비선형
        # 반응(뜨거울수록 냉각이 가속되는 것 같은 효과)을 못 잡아서, 그대로 두면
        # 거의 모든 이벤트에서 T 가 1등으로 나온다(실측: 8개 탱크 전부 80~100%)
        # — 진짜 보고 싶은 액추에이터 이상 신호가 그 밑에 묻힌다. 그래서
        # "주요차이요인" 후보에서 T 를 뺀다. T 의 차이값 자체는 diff_rows 표에
        # Δ_T 로 그대로 남겨서, 얼마나 컸는지는 여전히 볼 수 있게 한다.
        actionable = {f: v for f, v in diffs.items() if f != "T"}
        top_factor = max(actionable, key=lambda k: abs(actionable[k]))
        rows.append({
            "시각": lin["test_index"][i], "실제ΔT": round(float(lin["actual"][i]), 2),
            "선형예측": round(float(lin["pred"][i]), 2), "선형잔차": round(float(lin_resid[i]), 2),
            "부스팅예측": round(float(gb["pred"][i]), 2), "부스팅잔차": round(float(gb_resid[i]), 2),
            "잔차개선%": round((1 - abs(gb_resid[i]) / max(abs(lin_resid[i]), 1e-6)) * 100, 0),
            "주요차이요인(T제외)": top_factor, "차이값": round(actionable[top_factor], 2),
            "T_차이(참고용)": round(diffs["T"], 2),
        })
        diff_rows.append({"시각": lin["test_index"][i], **{f"Δ_{f}": round(diffs[f], 2) for f in feats}})
    return pd.DataFrame(rows), pd.DataFrame(diff_rows)


report, diffs = attribution_report(TANK, top_k=15)
report

**읽는 법.** `잔차개선%` 이 크게 양수면(부스팅이 많이 나아졌으면) `주요차이요인`
을 신뢰할 만하다 — 선형이 그 요인의 효과를 놓쳤다는 뜻이다. `잔차개선%` 이
0 근처거나 음수면, 부스팅도 못 맞춘 것이라 **이 4개 요인 밖의 원인**(센서
이상, 레시피 변경, 이 데이터에 없는 다른 액추에이터 등)일 가능성이 크다 —
그런 이벤트는 요인 이름이 나와도 믿으면 안 된다.

### ⚠️ 실행해보고 발견해서 고친 것 — T 를 후보에서 뺐다

처음엔 T(현재 온도)도 다른 요인들과 같이 랭킹에 넣었는데, 돌려보니 **8개
탱크 전부에서 T 가 80~100% 비율로 1등**이었다. T 는 액추에이터가 아니라
상태값이라 이건 유의미한 발견이 아니라 **선형모델이 T 의 비선형 반응을
못 잡는다는 모델링 한계**였다 — 실제 원인 규명에 방해만 됐다.

**T 를 후보에서 빼고 다시 보니** 탱크마다 뚜렷하게 한 액추에이터로 쏠렸다
(개선 이벤트 기준): P1·P5·P4·I2 는 **exchanger**, I3 는 **feed**, P3 는
**heat** 이 압도적이었다. 이게 훨씬 원래 목적(어느 액추에이터가 이상인지)에
가깝다. 아래 `주요차이요인(T제외)` 열과 7번 섹션의 `쏠림비율%` 이 이 기준으로
다시 계산된다.

In [ ]:
# 요인별 차이(Δ) 전체 표 — 어느 요인이 반복적으로 크게 벌어지는지 한눈에
diffs

## 5. 한 이벤트 자세히 보기 — 선형 vs SHAP 기여도 나란히

In [ ]:
EVENT_ROW = 0   # 위 report 표의 몇 번째 행을 볼지

lin, gb = FITS[TANK]
i = np.argsort(-np.abs(lin["actual"] - lin["pred"]))[EVENT_ROW]
lc = linear_contrib(lin, i)
feats = gb["features"]
shap_row = gb["shap"][i]

fig, ax = plt.subplots(figsize=(9, 4.5))
y_pos = np.arange(len(feats))
w = 0.35
ax.barh(y_pos - w/2, [lc[f] for f in feats], height=w, color="#1f77b4", label="선형 기여도")
ax.barh(y_pos + w/2, shap_row, height=w, color="#2ca02c", label="SHAP 기여도")
ax.set_yticks(y_pos); ax.set_yticklabels(feats)
ax.axvline(0, color="#888888", linewidth=0.8)
ax.set_title(f"{TANK} — {lin['test_index'][i]}  실제ΔT={lin['actual'][i]:.2f}  "
             f"(선형예측 {lin['pred'][i]:.2f} / 부스팅예측 {gb['pred'][i]:.2f})")
ax.legend(fontsize=9); ax.grid(axis="x", alpha=0.3)
fig.tight_layout()

## 6. 요인 전체 중요도 — 선형 |계수| 순위 vs SHAP 평균 크기 순위

개별 이벤트가 아니라 **테스트 구간 전체**에서 각 요인이 평균적으로 얼마나
중요했는지 비교한다. 두 순위가 크게 다르면, 그 요인은 상황에 따라 효과가
달라지는(비선형·상호작용) 요인이라는 뜻이다.

In [ ]:
lin, gb = FITS[TANK]
lin_importance = {f: abs(b) for f, b in zip(lin["features"][1:], lin["beta"][1:])}
shap_importance = {f: float(np.mean(np.abs(gb["shap"][:, j]))) for j, f in enumerate(gb["features"])}

imp = pd.DataFrame({
    "선형|계수|": lin_importance,
    "SHAP평균|기여도|": shap_importance,
})
imp["선형순위"] = imp["선형|계수|"].rank(ascending=False).astype(int)
imp["SHAP순위"] = imp["SHAP평균|기여도|"].rank(ascending=False).astype(int)
imp = imp.sort_values("SHAP평균|기여도|", ascending=False)
imp

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = imp.index
x = np.arange(len(order))
ax2 = ax.twinx()
ax.bar(x - 0.2, imp.loc[order, "선형|계수|"], width=0.4, color="#1f77b4", label="선형 |계수|")
ax2.bar(x + 0.2, imp.loc[order, "SHAP평균|기여도|"], width=0.4, color="#2ca02c", label="SHAP 평균|기여도|")
ax.set_xticks(x); ax.set_xticklabels(order, rotation=30, ha="right")
ax.set_ylabel("선형 |계수| (단위: ΔT/비율)", color="#1f77b4")
ax2.set_ylabel("SHAP 평균|기여도|", color="#2ca02c")
ax.set_title(f"{TANK} — 요인 중요도: 선형 vs SHAP (척도가 달라 값 자체보다 순위를 비교)")
fig.tight_layout()

## 7. 모든 탱크에 일괄 적용 — 각 탱크에서 가장 자주 등장하는 "주요차이요인"

탱크마다 반복해서 나오는 요인이 있다면, 그 탱크는 특정 요인의 비선형 효과가
선형 모델로 잘 안 잡힌다는 뜻이다.

In [ ]:
summary_rows = []
for tk in FITS:
    rep, _ = attribution_report(tk, top_k=15)
    if rep.empty:
        continue
    improved = rep[rep["잔차개선%"] > 20]     # 부스팅이 의미있게 나아진 이벤트만 신뢰
    vc = improved["주요차이요인(T제외)"].value_counts()
    top_factor = vc.index[0] if len(vc) else "-"
    concentration = round(vc.iloc[0] / len(improved) * 100, 0) if len(vc) else None
    summary_rows.append({
        "탱크": tk,
        "잔차TOP15_평균개선%": round(rep["잔차개선%"].mean(), 0),
        "신뢰가능이벤트수(개선>20%)": len(improved),
        "가장흔한주요요인(T제외)": top_factor,
        "쏠림비율%": concentration,
    })
pd.DataFrame(summary_rows)

## 8. 검증 — 가짜(합성) 고장을 주입해서, 이 방법이 진짜 원인을 찾아내는가

지금까지는 "탱크마다 한 요인으로 쏠린다"는 것만 확인했지, **그 요인이 실제로
맞는지는 확인한 적이 없다.** 실제 고장 사례가 데이터에 없어서(전부 정상
가동 중), 정답을 아는 합성 데이터를 만들어 검증한다.

### 만드는 방법

실제 탱크의 특징값(T, heat/exch/feed 의 ON 비율 — 즉 실제 운전 패턴)은 그대로
쓰고, **온도 변화(y)만 우리가 정한 공식으로 새로 만든다.** 대부분은 평범한
선형식이고, **딱 하나의 액추에이터에만** 온도가 특정 임계값을 넘었을 때만
켜지는 "문턱 효과"(선형 모델이 원리적으로 못 잡는 비선형)를 심는다.

```
y = -0.3·T + 5·heat_fut + 2·heat_past
          - 8·exch_fut  - 3·exch_past
          + 10·feed_fut + 4·feed_past
          + k · (문턱 효과를 심을 요인의 fut) · (T > 임계값)     ← 여기 하나만 비선형
          + 노이즈
```

그 다음 지금까지 쓴 파이프라인(선형 fit_eval, 부스팅 fit_gb, T 제외
attribution_report)을 **완전히 동일하게** 이 합성 데이터에 돌린다. 문턱 효과를
심은 요인이 "주요차이요인(T제외)" 1위로 나오면 맞힌 것이다.

**음성 대조군도 같이 돈다** — 비선형을 아예 안 심은(전부 선형인) 데이터에서는
특정 요인으로 쏠리면 안 된다(쏠리면 이 방법 자체가 헛것을 본다는 뜻).

In [ ]:
def make_synthetic(df, true_actuator=None, k=15.0, threshold=None, noise_sd=1.5, seed=0):
    """실제 특징 분포는 유지하고 y 만 합성한다. true_actuator=None 이면 순수 선형(음성 대조군)."""
    rng = np.random.default_rng(seed)
    T = df["T"].values
    if threshold is None:
        threshold = np.nanmedian(T)

    y = (-0.3 * T
         + 5 * df["heat_fut"].values + 2 * df["heat_past"].values
         - 8 * df["exch_fut"].values - 3 * df["exch_past"].values
         + 10 * df["feed_fut"].values + 4 * df["feed_past"].values)

    if true_actuator is not None:
        y = y + k * df[f"{true_actuator}_fut"].values * (T > threshold)

    out = df.copy()
    out["y"] = y + rng.normal(scale=noise_sd, size=len(y))
    return out


def detect_rate(synth_df, tank_label, true_actuator, top_k=15):
    """합성 데이터에 파이프라인을 그대로 돌려서, 잔차TOP-k 중 정답 요인 적중률을 잰다."""
    lin = ARX.fit_eval(synth_df, FEATURES)
    gb = fit_gb(synth_df, FEATURES)

    lin_resid = lin["actual"] - lin["pred"]
    order = np.argsort(-np.abs(lin_resid))[:top_k]

    hits, factors = 0, []
    for i in order:
        lc = linear_contrib(lin, i)
        shap_row = gb["shap"][i]
        actionable = {f: float(shap_row[j] - lc[f]) for j, f in enumerate(gb["features"]) if f != "T"}
        top = max(actionable, key=lambda k: abs(actionable[k]))
        factors.append(top)
        if true_actuator is not None and top.startswith(true_actuator):
            hits += 1

    from collections import Counter
    return {"탱크": tank_label, "실제주입요인": true_actuator or "(없음-대조군)",
            "부스팅개선%": round((1 - gb["rmse"] / lin["rmse"]) * 100, 1),
            "TOP15중_정답적중": hits if true_actuator else None,
            "적중률%": round(hits / top_k * 100, 0) if true_actuator else None,
            "실제나온요인분포": dict(Counter(factors))}

In [ ]:
VALIDATION_TANKS = ["I3", "P5"]     # 특성이 다른 두 탱크에서 반복 (성격이 다른 duty cycle)

val_rows = []
for tk in VALIDATION_TANKS:
    df, _ = ARX.build_frame(tk)
    for true_act in ["heat", "exch", "feed"]:
        synth = make_synthetic(df, true_actuator=true_act, seed=hash((tk, true_act)) % 10000)
        val_rows.append(detect_rate(synth, tk, true_act))
    # 음성 대조군 — 비선형을 아예 안 심었을 때
    synth0 = make_synthetic(df, true_actuator=None, seed=hash((tk, "none")) % 10000)
    val_rows.append(detect_rate(synth0, tk, None))

validation = pd.DataFrame(val_rows)
validation

### 어떻게 읽나

- **적중률%** 이 높으면(우연 수준인 33%[3개 요인 중 하나] 보다 훨씬 위) 이
  방법이 실제로 맞는 요인을 찾아낸다는 뜻이다.
- **대조군("실제주입요인"=없음)** 은 적중률 자체가 없고, 대신 `실제나온요인분포`
  가 특정 요인에 쏠리지 않고 고르게 퍼져 있어야 한다 — 쏠려 있다면 이 방법이
  아무 이상이 없어도 헛것을 가리킨다는 뜻이라 신뢰할 수 없다.
- 탱크(I3·P5)마다 실제 운전 패턴(듀티비, 사이클 속도)이 다른데도 비슷한
  적중률이 나오면, 특정 탱크의 우연이 아니라 방법 자체가 일반적으로 동작한다는
  근거가 된다.

## 9. 한계

- **SHAP 은 "부스팅 모델이 무엇에 의존했나"를 정확히 분해하지, 물리적 인과를
  증명하지 않는다.** 모델이 학습한 상관관계일 뿐이다.
- **잔차개선%이 낮은 이벤트에서 나온 "주요차이요인"은 신뢰하면 안 된다.**
  부스팅도 못 맞췄다는 건 이 4개 요인(T, heat, exchanger, feed)만으로는
  설명이 안 된다는 뜻이다 — 수위, 알람 이력, 다른 탱크와의 상호작용처럼
  지금 안 넣은 변수가 원인일 수 있다.
- **부스팅이 선형보다 안 나은 탱크는 이 분석 자체가 무의미하다** (2번 표에서
  RMSE개선%이 0 이하인 탱크).
- **표본 수가 적은 탱크(P4·I2)는 부스팅이 과적합하기 쉽다.** ARX 홀드아웃
  검증에서 학습 이벤트가 부족했던 탱크와 겹친다.

## 10. 재실행

```bash
python Tank_Temp_Factor_Analysis.py --start "2026-09-09 19:00:00"   # 데이터 갱신
```
그 다음 이 노트북을 처음부터 다시 실행하면 최신 데이터로 갱신된다.
`ARX_Temp_Model.py` 를 따로 돌릴 필요는 없다 — 이 노트북이 `build_frame` 과
`fit_eval` 을 직접 호출해서 매번 새로 학습한다.